# **AI-Based Energy Consumption Forecasting for Sustainable Energy Management**
### **Module:** Essentials and Applications of Artificial Intelligence - UFCE3P-30-3
### Assessment 2 — Group Project

## **1. Problem Statement:**

The stability of the electrical grid depends on maintaining a real-time balance between electricity supply and demand. Since large-scale storage is limited, accurate short-term demand forecasting is essential.

Forecasting errors create asymmetric risks:

* **Over-generation:** Increases operational cost and carbon emissions
* **Under-generation:** Risks grid instability and potential blackouts
* **Strategic limitation:** Reduces effective integration of renewable energy sources

**Objective:**
This project develops AI models to forecast hourly electricity demand in the AEP (American Electric Power) region using historical load and weather data.

---

## **2. Project Scope & Guardrails**
| ✅ In-Scope                                                         | ❌ Out-of-Scope                  |
| ------------------------------------------------------------------ | ------------------------------- |
| Hourly demand forecasting (2004–2018)                              | Full smart grid optimisation    |
| Weather impact analysis                                            | Renewable energy dispatch       |
| Model comparison (Baseline, Random Forest, XGBoost, optional LSTM) | National-level modelling        |
| Error analysis and failure cases                                   | Real-time production deployment |

> This project focuses on demand forecasting only and does not attempt full grid optimisation.

---

## **3. Core Research Questions**

1. Can hourly electricity demand be predicted with acceptable accuracy (MAPE in the range of 5–8%)?
2. Which model provides the best balance between accuracy and simplicity?
3. Do weather variables significantly improve prediction performance?
4. Under which conditions does the model fail (e.g., extreme weather, holidays)?

---

## **4. Mathematical Formulation**

This is a **supervised autoregressive regression problem with exogenous inputs (ARX)**:

$$\hat{y}_{t+1} = f\left(\underbrace{y_{t-1}, y_{t-24}, y_{t-168}}_{\text{lag features}},\ \underbrace{hour_t, day_t, month_t}_{\text{time features}},\ \underbrace{temp_t, humidity_t, \ldots}_{\text{weather features}}\right) + \epsilon_t$$

where $y_t$ = electricity demand (MW), $f(\cdot)$ = the learned model, $\epsilon_t$ = irreducible error.


---

## **5. The Data Foundation**

| Dataset                     | Records | Period    | Description                          |
| --------------------------- | ------- | --------- | ------------------------------------ |
| `AEP_hourly.csv`            | 121,273 | 2004–2018 | Historical electricity demand (PJM)  |
| `noaa_weather_columbus.csv` | 127,896 | 2004–2018 | Weather data (temperature, humidity) |

* Electricity data sourced from PJM Interconnection (via Kaggle)
* Weather data obtained using Open-Meteo API (derived from NOAA archives)

---

## **6. Evaluation Strategy**

* Train-test split based on time (no random shuffling)
* Metrics used:

  * Mean Absolute Error (MAE)
  * Root Mean Squared Error (RMSE)
  * Mean Absolute Percentage Error (MAPE)

## **7. Environment Setup**
This section initialises the project environment by importing all required libraries, defining file paths, and setting a fixed random seed to ensure reproducibility. Optional dependencies (e.g., TensorFlow, statsmodels) are handled using conditional imports to allow the notebook to run even if they are not installed. A consistent plotting style and output directory are also configured.

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, random

warnings.filterwarnings("ignore")

# Global Configuration
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))  # go up one level

DATA_DIR = os.path.join(BASE_DIR, "data")

ENERGY_PATH  = os.path.join(DATA_DIR, "AEP_hourly.csv")
WEATHER_PATH = os.path.join(DATA_DIR, "noaa_weather_columbus.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Plot style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Optional Libraries
try:
    import statsmodels.api as sm
    print("statsmodels ready.")
except ImportError:
    sm = None
    print("statsmodels not available.")

try:
    import tensorflow as tf
    tf.random.set_seed(RANDOM_SEED)
    print("TensorFlow 2.x ready.")
except ImportError:
    tf = None
    print("TensorFlow not available.")

# Utility
def save_fig(name):
    path = os.path.join(OUTPUT_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")

# Status
print("\nEnvironment ready.")
print(f"  Energy  : {ENERGY_PATH}")
print(f"  Weather : {WEATHER_PATH}")
print(f"  Outputs : {OUTPUT_DIR}")
print(f"  Seed    : {RANDOM_SEED}")

statsmodels ready.
TensorFlow not available.

Environment ready.
  Energy  : c:\Users\tahermamun\Desktop\EAAI - Project\data\AEP_hourly.csv
  Weather : c:\Users\tahermamun\Desktop\EAAI - Project\data\noaa_weather_columbus.csv
  Outputs : c:\Users\tahermamun\Desktop\EAAI - Project\outputs
  Seed    : 42


## **8. Data Loading**

In [2]:
energy_raw  = pd.read_csv(ENERGY_PATH)
weather_raw = pd.read_csv(WEATHER_PATH)

print("=" * 58)
print("ENERGY DATASET — AEP_hourly.csv")
print(f"  Shape   : {energy_raw.shape[0]:,} rows × {energy_raw.shape[1]} columns")
print(f"  Columns : {list(energy_raw.columns)}")
print(f"  MW range: {energy_raw['AEP_MW'].min():,.0f} – {energy_raw['AEP_MW'].max():,.0f} MW")
print("=" * 58)
print("WEATHER DATASET — noaa_weather_columbus.csv")
print(f"  Shape   : {weather_raw.shape[0]:,} rows × {weather_raw.shape[1]} columns")
print(f"  Columns : {list(weather_raw.columns)}")
print(f"  Temp range: {weather_raw['temp_C'].min():.1f}°C – {weather_raw['temp_C'].max():.1f}°C")
print("=" * 58)
display(energy_raw.head(3))
display(weather_raw.head(3))

ENERGY DATASET — AEP_hourly.csv
  Shape   : 121,273 rows × 2 columns
  Columns : ['Datetime', 'AEP_MW']
  MW range: 9,581 – 25,695 MW
WEATHER DATASET — noaa_weather_columbus.csv
  Shape   : 127,896 rows × 6 columns
  Columns : ['datetime', 'temp_C', 'humidity', 'wind_speed', 'pressure', 'precipitation']
  Temp range: -28.9°C – 38.1°C


,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0


,datetime,temp_C,humidity,wind_speed,pressure,precipitation
0,2004-01-01 00:00:00,0.1,86,7.6,998.2,0.0
1,2004-01-01 01:00:00,-0.5,89,7.9,998.5,0.0
2,2004-01-01 02:00:00,-1.0,90,8.6,999.1,0.0


## **9. Basic Data Understanding**
**Data Quality Audit & Profiling**
To ensure a robust analysis, we performed an automated data profiling step using ydata-profiling. This is essential for identifying underlying data issues that basic manual inspection.

In [3]:
# run once to install the profiling tool
%pip install ydata-profiling -q

Note: you may need to restart the kernel to use updated packages.


In [6]:
from ydata_profiling import ProfileReport

# Setup paths (robust)
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPORT_DIR = os.path.join(BASE_DIR, "y_report")
os.makedirs(REPORT_DIR, exist_ok=True)

# Function to generate report
def generate_report(df, name):
    report_path = os.path.join(REPORT_DIR, f"{name}.html")
    
    if not os.path.exists(report_path):
        print(f"Generating {name} report...")
        
        profile = ProfileReport(
            df,
            title=f"{name} Data Audit",
            minimal=True
        )
        
        profile.to_file(report_path)
        print(f"Saved: {report_path}")
    else:
        print(f"{name} report already exists.")

# Generate reports
generate_report(energy_raw, "aep_data_audit")
generate_report(weather_raw, "weather_data_audit")

Generating aep_data_audit report...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:01<00:00,  1.70it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 6/6 [00:01<00:00,  4.39it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: c:\Users\tahermamun\Desktop\EAAI - Project\y_report\weather_data_audit.html
